# Latent Space Interpretation

This notebook is a lightweight public-facing front end for the cleaned
latent-space interpretation workflow. The original `interpret_embeddings.ipynb`
notebook is preserved separately as an exploratory record.


## Setup

Run this cell first so the notebook executes from the repository root and uses
the active kernel's Python interpreter.


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.executable}")


## Build PCA Summaries

This script loads the learned-embedding split tables for each head, fits per-head
PCA bases, measures the manuscript-reported local PC0 stability against HL-gap
and the target, and writes the reference PCA basis used by the interpretation workflow.


In [ ]:
!{sys.executable} scripts/analysis/build_interpretation_tables.py \
  --target abs \
  --target-col peakwavs_max \
  --dataset deep4chem \
  --split scaffold \
  --N N11816 \
  --device cpu \
  --embedding-cache-root local_runs/latent_interpretation_cache \
  --outdir figures/latent_interpretation


Expected outputs:

- `figures/latent_interpretation/pca_explained_variance.csv`
- `figures/latent_interpretation/pc0_stability_summary.csv`
- `figures/latent_interpretation/pc0_stability_aggregate.json`
- `figures/latent_interpretation/pca_cumulative_variance.pdf`
- `figures/latent_interpretation/reference_pca_basis.joblib`


## Feature Importance Stability

This script fits downstream XGBoost models on PCA-compressed latent features plus
solvent fingerprints and compares feature-importance stability across heads.


In [ ]:
!{sys.executable} scripts/analysis/plot_feature_importance_stability.py \
  --target abs \
  --target-col peakwavs_max \
  --dataset deep4chem \
  --split scaffold \
  --N N11816 \
  --device cpu \
  --embedding-cache-root local_runs/latent_interpretation_cache \
  --outdir figures/latent_interpretation


Expected outputs:

- `figures/latent_interpretation/feature_importance_stability.csv`
- `figures/latent_interpretation/feature_importance_consensus.csv`
- `figures/latent_interpretation/feature_importance_stability.pdf`


## PC-to-Property Heatmap

This script orders important principal components by consensus XGBoost importance,
aligns signs across heads, and plots their correlations with selected physical descriptors.


In [ ]:
!{sys.executable} scripts/analysis/plot_pc_property_heatmap.py \
  --target abs \
  --target-col peakwavs_max \
  --dataset deep4chem \
  --split scaffold \
  --N N11816 \
  --device cpu \
  --embedding-cache-root local_runs/latent_interpretation_cache \
  --corr-method spearman \
  --consensus-csv figures/latent_interpretation/feature_importance_consensus.csv \
  --outdir figures/latent_interpretation


Expected outputs:

- `figures/latent_interpretation/pc_property_heatmap.csv`
- `figures/latent_interpretation/pc_property_heatmap.pdf`


## Latent Density Panels

This script reproduces the Supporting Information hexbin panel figure linking selected
principal components to wavelength and descriptor trends.


In [ ]:
!{sys.executable} scripts/analysis/plot_latent_density_panels.py \
  --target abs \
  --target-col peakwavs_max \
  --dataset deep4chem \
  --split scaffold \
  --N N11816 \
  --head 1 \
  --outdir figures/latent_interpretation


Expected outputs:

- `figures/latent_interpretation/latent_density_panels.png`


## SHAP Analysis

This script fits the head-specific PCA-XGB member-00 model, computes SHAP values on a
training subsample, and writes both the basic summary plot and the two-panel composite figure.


In [ ]:
!{sys.executable} scripts/analysis/plot_shap_latent_analysis.py \
  --target abs \
  --target-col peakwavs_max \
  --dataset deep4chem \
  --split scaffold \
  --N N11816 \
  --head 1 \
  --outdir figures/latent_interpretation


Expected outputs:

- `figures/latent_interpretation/shap_summary_plot.png`
- `figures/latent_interpretation/shap_composite_plot.png`


## Large Latent-Space Plots

This script generates the large `PC0` vs target manifold plot and the `PC0` vs `PC8`
latent map used in the interpretation SI. It also writes the associated representative
molecule PNGs for local inspection.


In [ ]:
!{sys.executable} scripts/analysis/plot_latent_maps.py \
  --target abs \
  --target-col peakwavs_max \
  --dataset deep4chem \
  --split scaffold \
  --N N11816 \
  --head 1 \
  --outdir figures/latent_interpretation \
  --molecule-dir figures/latent_interpretation/molecule_examples


Expected outputs:

- `figures/latent_interpretation/pc0_vs_target_manifold.png`
- `figures/latent_interpretation/pc0_vs_pc8_map.png`
- `figures/latent_interpretation/molecule_examples/`
